## What this notebook does
This notebook builds a lightweight RAG pipeline for Risalo search: normalize the query, expand key themes, embed the text, retrieve nearest verses, and return structured context.

# RAG Retrieval Notebook
A compact retrieval pipeline for Shah Abdul Latif Bhittai verse search.

In [154]:
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import re

## Embedding Model
We use a multilingual SentenceTransformer so Roman Sindhi queries and verse text share the same vector space. That makes semantic retrieval work even when the query wording is different from the source text.

In [155]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10571.29it/s]


## Handling misspellings and variations
To handle misspellings and variations in Roman Sindhi, we can implement a simple normalization function that maps common misspellings to their correct forms. This can be done using a dictionary of known misspellings and their corrections.

Example Roman Sindhi misspellings:
1. "ishq" might be misspelled as "ishk" or "ishaq"
2. "chayo" might be misspelled as "chayyo" or "chaayo"
3. "ayen" might be misspelled as "ain" or "ayen"
4. "muhnja" might be misspelled as "munja" or "muhnjaa"


In [156]:
def preprocess_roman_sindhi(text):
    if not text:
        return ""

    text = text.lower().strip()

    normalization_map = {
        r"ai": "e",
        r"ay+": "e",
        r"ey+": "e",
        r"iy+": "i",
        r"jh": "j",
        r"bh": "b",
        r"kh": "k",
        r"aa+": "a",
        r"ee+": "i",
        r"oo+": "u",
        r"uu+": "u",
        r"aan\b": "an",
        r"oon\b": "un",
    }
    for pattern, replacement in normalization_map.items():
        text = re.sub(pattern, replacement, text)

    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)

    return text

## Expand Query

Since the characters within Shah Latif poetry are not embedded properly as a proper noun therefore we are using synonyms and related terms to expand the query. For example, if the query is about "Noori" we can expand it to include related terms like Noori Jam Tamachi Keenjhar Kamod Fisherwoman Lake Humility Lowly-born King-Jam-Tamachi Gandri etc. This helps in retrieving relevant verses that may not contain the exact term "Noori" but are still relevant to the query.

In [157]:
def expand_query(user_query):
    synonyms = {
        "noori": "Noori Jam Tamachi Keenjhar Kamod Fisherwoman Lake Humility Lowly-born King-Jam-Tamachi Gandri",
        "sassui": "Sassui Punnun Punhoo Kech Makran Mountain Desert Jabal Husseini Desi Abri Maazoori Kohiyari Quest Struggle Grief Separation",
        "sohni": "Sohni Mehar Izzat-Baqi River Dariya Sahar Jar Drowning Union Gharrho Pitcher Dam-Sahar Mehran",
        "marui": "Marui Malir Tharparkar Umar Umar-Kot Marvi Patriotism Prison Homeland Chains Khenrri Desert-Life Loee Paneer Sanghar Dhat Dhatiyun Cattles Village Folks Panooharr",
        "umar": "Umar Umar-Kot King Tyrant Prison Chains Castle Fort Palace Marui Malir Tharparkar",
        "malir": "Malir Homeland Tharparkar Desert Village Marui Maru Sanghar Panooharr Peelu Greenery Rain",
        "lila": "Leela or Lilan Chanesar Dasal Diamond Necklace Haar Greed Regret Vanity Kaunru Jewelry Repentance Doso",
        "momal": "Momal Rano Kak-Mahal Magic Doubt Trust Reconciliation Sodha Mehran Gujar-Rano Luddano",
        "lila": "Lilan Chanesar Dasal Diamond Necklace Greed Regret Vanity Kaunru Jewelry Repentance",
        "sorath": "Sorath Rai Diyach Sacrifice Head Music King Charity Minstrel Bijal Junagarh Strings Surando",
        "nuri_jam_tamachi": "Kamod Noori Jam Tamachi Keenjhar Fisherwoman Gandri Humility Royal-Love King-Jam",
        "sasui_punhun": "Sassui Punnun Kech Caravan Camel-riders Brothers Ari-Jam Hot Baloch Kechi",
        "kalyan": "Divine Love Oneness Peace Healing Physician Arrow Treatment Remedy Medicine Hakim",
        "yaman_kalyan": "Self-Purification Patient Seeker Spirit Sacrifice Agony Burning Wound Fire",
        "khambhat": "Moon Stars Camel Journey Prophet Messenger Beauty Caravan Night-Traveler Karvan",
        "sri_raag": "Merchant Ship Ocean Pearl Wealth Spiritual Trade Sailors Voyage Deep-Sea Shore",
        "samundi": "Sailor Separation Longing Coast Voyage Return Sea-Trade Wives Waiting Ships",
        "maazoori": "Weakness Exhaustion Footprints Grief Loneliness Wanderer Crippled Limping Helpless",
        "desi": "Kech Caravan Brothers Separation Camel-rider Heartache Ari-Jam Hot-Baloch",
        "kohiyari": "Rocky Mountains Path Fatigue Beloved Footsteps Hardship Stone Steep-Pass",
        "husseini": "Martyrdom Sacrifice Karbala Thirst Patience Agony Imam Hussain Abbas Yazid Battle",
        "kamod": "Keenjhar Lake Noori Fisherwoman Lowly Humility Sur-Kamod King-Tamachi",
        "ghatu": "Fishermen Whale Kalachi Fear Courage Sacrifice Deep-Sea Nets Fishing-Brave Whirlpool",
        "kedaro": "Heroism Battle Struggle Martyr Death Honor Battlefield Sword Blood Muharram",
        "sarang": "Rain Monsoon Clouds Water Life Joy Purity Farmers Thunder Lightening Greenery",
        "asa": "Hope Longing Mirror Vision Silence Beloved Formless Spirit Reflection",
        "ripar_bhar": "Separation Grief Lament Loneliness Despair Tears Suffering Agony",
        "khahori": "Ascetics Jogis Seeking Truth Spiritual-Path Renunciation Dust Mountains Hermits",
        "ramkali": "Jogis Sanyasis Meditation Ashes Seeking God Guru Monastery Begging-Bowl Silence",
        "kapaiti": "Spinner Thread Weaving Dedication Purity Hard-work Cotton Loom Spinning-Wheel",
        "purab": "East Sun Worship Morning Devotion Spiritual-Light Sanyasis Orient",
        "karayal": "Swan Lake Purity Corruption Water Lotus Soul Crane Hunter Muddy-Water",
        "pirbhati": "Morning-Light Dawn Prayer Spiritual-Awakening Sunrise Sapna Devotee",
        "dahar": "Desert Barren Thirst Endurance Resilience Solitude Arid-Land Sand-Dunes",
        "bilawal": "Samma Charity Bravery Royalty Generosity Honor Jakhro King Raja Hero",
        "ishq": "Love Divine Kalyan Yaman Burning Passion Union Beloved Sacrifice",
        "dariya": "River Ocean Water Sea Sohni Samundi Ghatu Crossing Boat Tide Deep",
    }

    expanded_query = user_query.lower()
    for key, val in synonyms.items():
        if key in expanded_query:
            expanded_query = f"{val} | {expanded_query}"
    return expanded_query

## Setup and Indexing
Load the model, prepare the corpus, and build the FAISS index.

In [158]:
with open('formatted_dataset/risalo.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

corpus = [preprocess_roman_sindhi(item['searchable_text']) for item in data if item['content'].get('explanation', '').strip()]

print(f"Total documents in dataset: {len(data)}")
print(f"Documents with non-empty explanations: {len(corpus)}")

Total documents in dataset: 4730
Documents with non-empty explanations: 2519


In [159]:
print('Generating embeddings... this might take a moment.')
embeddings = model.encode(corpus, show_progress_bar=True, convert_to_numpy=True)

print(f'Embeddings shape: {embeddings.shape}')

Generating embeddings... this might take a moment.


Batches: 100%|██████████| 79/79 [00:45<00:00,  1.74it/s]

Embeddings shape: (2519, 384)


In [160]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype('float32'))
print(f'Total items in FAISS index: {index.ntotal}')

Total items in FAISS index: 2519


# Example usage
Run a sample queries and print the retrieved context.

In [161]:
queries = [
    "Sacha ishq bare mein chah chayo aahe?",
    "Jabal ain Sassui jo dukha",
    "Keenjhar jo kinaro ain Noori",
    "Malir ain Marui ji galh",
    "Dariya ain dunya",
    "Allah hidayat kare",
    "Sohni Mehar ain gharrho",
    "Umar ain Marui ji katcheri",
    "Mumal Rano ain Kak Mahal",
    "Laila Chanesar ain haar",
    "Sorath jo sir ain Rai Diyach",
    "Karbala ain Imam Hussain",
    "Hussain te cha zulm thio?",
    "Kangan jo kookar",
    "Sami ain Kapri",
    "Vanjara ain vopar",
    "Miskeen ain bhala manhoon",
    "Dahiri ain dunya jo fikr",
    "Punhoon jo pech",
    "Sarang ain meenhan ji dua",
    "Ghareeb ain ameer jo fark",
]

results = []
for query in queries:
    query_processed = preprocess_roman_sindhi(expand_query(query))
    query_vector = model.encode([query_processed])
    k = 5
    distances, indices = index.search(query_vector.astype('float32'), k)

    lines = [
        f"Query: {query}",
        f"Processed: {query_processed}",
        "Top matches:",
    ]

    for rank, (idx, distance) in enumerate(zip(indices[0], distances[0]), start=1):
        melody = data[idx]["metadata"]["melody"]
        lines.append(f"  {rank}. {melody}  |  distance: {float(distance):.4f}")

    results.append("\n".join(lines))

results = ("\n" + "-" * 80 + "\n").join(results)

print(results)

Query: Sacha ishq bare mein chah chayo aahe?
Processed: love divine kalyan yaman burning passion union beloved sacrifice sacha ishq bare mein chah cheo ahe
Top matches:
  1. سُر آبڙي  |  distance: 9.9273
  2. سُر يمن ڪلياڻ  |  distance: 11.2763
  3. سُر سھڻي  |  distance: 11.4167
  4. سُر سارنگ  |  distance: 11.5458
  5. سُر سارنگ  |  distance: 11.7591
--------------------------------------------------------------------------------
Query: Jabal ain Sassui jo dukha
Processed: sassui punnun punhu kech makran mounten desert jabal husseini desi abri mazuri kohiari quest struggle grief separation jabal en sassui jo duka
Top matches:
  1. سُر سامونڊي  |  distance: 2.1431
  2. سُر آبڙي  |  distance: 2.2049
  3. سُر ديسي  |  distance: 2.2526
  4. سُر سھڻي  |  distance: 2.2903
  5. سُر معذوري  |  distance: 2.3116
--------------------------------------------------------------------------------
Query: Keenjhar jo kinaro ain Noori
Processed: nuri jam tamachi kinjar kamod fisherwoman lake humility 

## Retrieval Helper
`search_risalo` returns structured retrieval context instead of a generated answer.

Examples:
- `search_risalo("Keenjhar jo kinaro ain Noori")`
- `search_risalo("Malir ain Marui ji galh")`
- `search_risalo("Karbala ain Imam Hussain")`

In [166]:
def search_risalo(user_query):
    processed_query = expand_query(user_query)
    # processed_query = preprocess_roman_sindhi(expand_query(user_query))
    query_vector = model.encode([processed_query])
    distances, indices = index.search(query_vector.astype('float32'), k=5)

    matches = []
    for idx in indices[0]:
        item = data[idx]
        matches.append(
            {
                'sur': item['metadata']['melody'],
                'verse': item['content']['roman_text'],
                'explanation': item['content']['explanation'],
            }
        )

    return {
        'user_query': user_query,
        'processed_query': processed_query,
        'distances': distances[0].tolist(),
        'matches': matches,
    }

## Retrieval Helper
`search_risalo` returns structured retrieval context instead of a generated answer.

Examples:
- `search_risalo("Keenjhar jo kinaro ain Noori")`
- `search_risalo("Malir ain Marui ji galh")`
- `search_risalo("Karbala ain Imam Hussain")`

In [167]:
context = search_risalo("Sacha ishq bare mein chah chayo aahe?")
print(json.dumps(context, indent=2, ensure_ascii=False))

{
  "user_query": "Sacha ishq bare mein chah chayo aahe?",
  "processed_query": "Love Divine Kalyan Yaman Burning Passion Union Beloved Sacrifice | sacha ishq bare mein chah chayo aahe?",
  "distances": [
    9.320545196533203,
    10.524017333984375,
    10.675968170166016,
    10.815369606018066,
    10.954211235046387
  ],
  "matches": [
    {
      "sur": "سُر آبڙي",
      "verse": "Waaquf'u na wannkaar'a jee, aggiyaan safar'u sattaanno,\nAaddaa lak'a Lateef'u chae, jabal'u joraanno,\nPehro ut'ay paanno, sawalo kij'ay saath'a jo.",
      "explanation": "(سسئي چوي ٿي) مون کي وڻڪار (جھڙي ڏکي جڳهہ) جي بلڪل ڪا خبر ڪانهي، هوڏانھن سفر سخت ڪٺن آهي. پھاڙن جا پيچرا هيٺ مٿي ۽ آڏا ترڇا آهن. عبداللطيف چوي ٿو ان کان سواءِ جبل ڏاڍو ڏکيو (ظالم) آهي. اي مالڪ سائين! تون ئي پھرين منزل کي سولو ۽ آسان ڪر."
    },
    {
      "sur": "سُر سارنگ",
      "verse": "Ochan ghar je ajhko, jhopo sahi na see,\nSuraenjh sorh khy, haal muhnjo hee,\nAgan ayo thi, ta dola kehn dang thiyan.",
      "explanation": "س

## Save the FAISS Index
`faiss.write_index` serializes the built vector index to disk so the retrieval setup can be reused without recomputing embeddings every time.

In [164]:
faiss.write_index(index, 'risalo_index.faiss')

## Evaluation
Test retrieval accuracy against curated ground-truth queries and expected Surs.

In [168]:
import numpy as np
import pandas as pd

test_cases = test_cases = [
    # --- SUR SARANG (Monsoon/Hope) ---
    ("meenhan barish ji umeed", "سُر سارنگ"),
    ("badal ain bijli jo dhabko", "سُر سارنگ"),
    ("sarang ain dharti ji khushali", "سُر سارنگ"),
    ("tarsayal thar ain barish", "سُر سارنگ"),
    ("paharun te meenhan ji dua", "سُر سارنگ"),
    # --- SUR SOHNI (Sacrifice/River) ---
    ("sohni ain mehar ji pirit", "سُر سھڻي"),
    ("gharro ain dariya ji dahat", "سُر سھڻي"),
    ("dum ain dushman ji dunya", "سُر سھڻي"),
    ("sahar ain jar mein judai", "سُر سھڻي"),
    ("ishq mein dubyal sohni", "سُر سھڻي"),
    # --- SUR MARUI (Patriotism/Captivity) ---
    ("marui ain malir jo kookar", "سُر مارئي"),
    ("umar kot ji qaid ain zanjeer", "سُر مارئي"),
    ("loee ain paneer jo maan", "سُر مارئي"),
    ("sanghar ain dhatiun jo ghum", "سُر مارئي"),
    ("vatan ji yaad mein rual marui", "سُر مارئي"),
    ("umar jo zulm ain marui ji himat", "سُر مارئي"),
    # --- SUR SASSUI (The Four Surs of the Quest) ---
    ("sassui punhun ji judai", "سُر حسيني"),
    ("jabal ain pathar jo safar", "سُر معذوري"),
    ("kech ja karwan ain bhotar", "سُر ديسي"),
    ("punhun ji pachar mein piral", "سُر ڪوھياري"),
    ("ari jam ja put ain sasui", "سُر ديسي"),
    ("pahirun ain gaddoja jo dard", "سُر معذوري"),
    # --- SUR KAMOD (Humility/Keenjhar) ---
    ("keenjhar dhand ain jam tamachi", "سُر ڪاموڏ"),
    ("noori gandri ain raja jo ishq", "سُر ڪاموڏ"),
    ("machhiyani ain tamachi ji tazzim", "سُر ڪاموڏ"),
    ("miani ain meenhan jo nazaro", "سُر ڪاموڏ"),
    # --- SUR KEDARO (Heroism/Karbala) ---
    ("karbala jo maidan ain imam hussain", "سُر ڪيڏارو"),
    ("yazid jo zulm ain shahadat", "سُر ڪيڏارو"),
    ("abbas ain asghar jo gham", "سُر ڪيڏارو"),
    ("deen laiy sir jo nazrano", "سُر ڪيڏارو"),
    ("muharram ji pachar ain matam", "سُر ڪيڏارو"),
    # --- SUR RAMKALI & PURAB (Ascetics) ---
    ("jogi ain sanyasi ji taryo", "سُر رامڪلي"),
    ("ashes ain dhunni jo dhuo", "سُر رامڪلي"),
    ("nani ain hinglaj jo safar", "سُر پورب"),
    ("guru ain chela ji katcheri", "سُر رامڪلي"),
    ("khahori ain parbat jo panth", "سُر کاھوڙي"),
    # --- SUR SORATH (Divine Sacrifice/Music) ---
    ("sorath ain rai diyach ji qurbani", "سُر سورٺ"),
    ("bijal jo chang ain suro", "سُر سورٺ"),
    ("junagarh jo raja ain mangno", "سُر سورٺ"),
    ("surando ain sir jo sawal", "سُر سورٺ"),
    # --- SUR RAANO (Doubt/Kak Mahal) ---
    ("mumal rano ain kak mahal jo jadu", "سُر راڻو"),
    ("luddano ain sodho jo firaq", "سُر راڻو"),
    ("dhola ain mormal ji pirit", "سُر راڻو"),
    # --- SUR LAILA CHANESAR ---
    ("laila ain dasal jo haar", "سُر ليلا"),
    ("chanesar ji chauk ain pachtao", "سُر ليلا"),
    ("kaunru ain gehna ji lalach", "سُر ليلا"),
    # --- MORAL & PHILOSOPHICAL (Kalyan/Asa) ---
    ("sacho ishq ain haqiqi mahboob", "سُر ڪلياڻ"),
    ("allah ji wahdaniyat ain tawheed", "سُر ڪلياڻ"),
    ("dunya fani aahe ain ghaflat", "سُر آسا"),
    ("insan jo nafas ain khudi", "سُر سھڻي"),
    ("pakeeza pirit ain qalb jo sukoon", "سُر يمن ڪلياڻ"),
]


def evaluate_retrieval(test_data, k_values=[1, 3, 5, 10]):
    results = []

    for query, expected_sur in test_data:
        processed_query = preprocess_roman_sindhi(query)
        expanded_query = expand_query(processed_query)

        query_vector = model.encode([expanded_query]).astype("float32")
        _, indices = index.search(query_vector, max(k_values))

        found_surs = [data[i]["metadata"]["melody"] for i in indices[0]]

        row = {"query": query, "expected": expected_sur}
        for k in k_values:
            row[f"Top-{k}"] = 1 if expected_sur in found_surs[:k] else 0
        results.append(row)

    df = pd.DataFrame(results)
    metrics = {f"Accuracy @{k}": df[f"Top-{k}"].mean() * 100 for k in k_values}

    return df, metrics


report_df, final_metrics = evaluate_retrieval(test_cases)

print("--- RAG RETRIEVAL PERFORMANCE ---")
for metric, val in final_metrics.items():
    print(f"{metric}: {val:.2f}%")

report_df.to_csv("retrieval_benchmark.csv", index=False)

--- RAG RETRIEVAL PERFORMANCE ---
Accuracy @1: 9.80%
Accuracy @3: 13.73%
Accuracy @5: 23.53%
Accuracy @10: 27.45%
